<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/modelTrain_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#═══════════════════════════════════════════
# STEP 1: Install packages
#═══════════════════════════════════════════
import os
os.environ["WANDB_DISABLED"] = "true"

!pip uninstall transformers accelerate peft sentence-transformers -y -q
!pip install transformers==4.40.0 accelerate==0.27.2 peft==0.9.0 -q

print("✅ Installation complete — Now restart runtime!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.0 MB/s eta 0:00:00
✅ Installation complete — Now restart runtime!


In [2]:
#═══════════════════════════════════════════
# STEP 3: Verify versions
#═══════════════════════════════════════════
import transformers, accelerate, peft

print(f"transformers : {transformers.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"peft         : {peft.__version__}")

ok = (
    transformers.__version__ == "4.40.0" and
    accelerate.__version__   == "0.27.2" and
    peft.__version__         == "0.9.0"
)

print("\n✅ Versions correct!" if ok else "\n❌ Wrong version — repeat STEP 1!")

transformers : 4.40.0
accelerate   : 0.27.2
peft         : 0.9.0

✅ Versions correct!


In [3]:
#═══════════════════════════════════════════
# STEP 4: Verify GPU
#═══════════════════════════════════════════
import torch

print(f"GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name      : {torch.cuda.get_device_name(0)}")
    print("\n✅ GPU active — training will be fast!")
else:
    print("\n⚠️  No GPU! Runtime → Change Runtime Type → GPU")

GPU Available : True
GPU Name      : Tesla T4

✅ GPU active — training will be fast!


In [4]:
#═══════════════════════════════════════════
# STEP 5: Upload CSV files
#═══════════════════════════════════════════
import os
import pandas as pd
from google.colab import files

os.environ["WANDB_DISABLED"] = "true"

print("Upload these 4 files:")
print("   1. train_combined.csv")
print("   2. val_combined.csv")
print("   3. test_combined.csv")
print("   4. unlabeled_dataset.csv")

uploaded = files.upload()

train_df     = pd.read_csv("train_combined.csv")
val_df       = pd.read_csv("val_combined.csv")
test_df      = pd.read_csv("test_combined.csv")
df_unlabeled = pd.read_csv("unlabeled_dataset.csv")

print(f"\n✅ Files loaded!")
print(f"   train_df     : {len(train_df):,} rows")
print(f"   val_df       : {len(val_df):,} rows")
print(f"   test_df      : {len(test_df):,} rows")
print(f"   df_unlabeled : {len(df_unlabeled):,} requirements")
print(f"\nPriority Distribution (Train):")
print(train_df['priority'].value_counts())

Upload these 4 files:
   1. train_combined.csv
   2. val_combined.csv
   3. test_combined.csv
   4. unlabeled_dataset.csv


Saving unlabeled_dataset.csv to unlabeled_dataset.csv
Saving test_combined.csv to test_combined.csv
Saving val_combined.csv to val_combined.csv
Saving train_combined.csv to train_combined.csv

✅ Files loaded!
   train_df     : 23,274 rows
   val_df       : 2,909 rows
   test_df      : 2,910 rows
   df_unlabeled : 97 requirements

Priority Distribution (Train):
priority
Medium    9430
High      8550
Low       5294
Name: count, dtype: int64


In [5]:
#═══════════════════════════════════════════
# STEP 6: Load TinyBERT
#═══════════════════════════════════════════
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "huawei-noah/TinyBERT_General_4L_312D"

print("Loading TinyBERT...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model     = BertForSequenceClassification.from_pretrained(
                model_name, num_labels=3
            )

print(f"\n✅ TinyBERT loaded!")
print(f"   GPU Active : {torch.cuda.is_available()}")
print(f"   Labels     : High(0)  Medium(1)  Low(2)")

Loading TinyBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ TinyBERT loaded!
   GPU Active : True
   Labels     : High(0)  Medium(1)  Low(2)


In [6]:
#═══════════════════════════════════════════
# STEP 7: Create dataset class
#═══════════════════════════════════════════
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score

class RequirementsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['priority_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation     = True,
            padding        = 'max_length',
            max_length     = self.max_len,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(
                                  self.labels[index],
                                  dtype=torch.long
                              )
        }

train_dataset = RequirementsDataset(train_df, tokenizer)
val_dataset   = RequirementsDataset(val_df,   tokenizer)
test_dataset  = RequirementsDataset(test_df,  tokenizer)

print("✅ Datasets created!")
print(f"   Train : {len(train_dataset):,}")
print(f"   Val   : {len(val_dataset):,}")
print(f"   Test  : {len(test_dataset):,}")

✅ Datasets created!
   Train : 23,274
   Val   : 2,909
   Test  : 2,910


In [7]:
#═══════════════════════════════════════════
# STEP 8: Class weights + Weighted Trainer
# Fixes Medium class low recall problem
#═══════════════════════════════════════════
from torch import nn
from transformers import Trainer
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1, 2]),
    y            = train_df['priority_id'].values
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:")
print(f"   High   (0) : {class_weights[0]:.4f}")
print(f"   Medium (1) : {class_weights[1]:.4f}")
print(f"   Low    (2) : {class_weights[2]:.4f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs,
                     return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")

        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

print("\n✅ WeightedTrainer ready!")

Class Weights:
   High   (0) : 0.9074
   Medium (1) : 0.8227
   Low    (2) : 1.4654

✅ WeightedTrainer ready!


In [8]:
#═══════════════════════════════════════════
# STEP 9: Metrics function (macro F1 as primary)
#═══════════════════════════════════════════
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    accuracy       = accuracy_score(labels, predictions)
    f1_macro       = f1_score(labels, predictions, average='macro')
    f1_weighted    = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy'    : round(accuracy,    4),
        'f1_score'    : round(f1_macro,    4),
        'f1_weighted' : round(f1_weighted, 4),
    }

print("✅ Metrics function ready!")
print("   Tracking: accuracy, f1_score (macro), f1_weighted")

✅ Metrics function ready!
   Tracking: accuracy, f1_score (macro), f1_weighted


In [9]:
#═══════════════════════════════════════════
# STEP 10: Improved training configuration
#═══════════════════════════════════════════
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                   = './tinybert-priority-final',

    num_train_epochs             = 10,
    per_device_train_batch_size  = 32,
    per_device_eval_batch_size   = 32,

    learning_rate                = 3e-5,
    weight_decay                 = 0.01,
    warmup_steps                 = 500,
    max_grad_norm                = 1.0,
    lr_scheduler_type             = 'cosine',

    evaluation_strategy          = 'epoch',
    save_strategy                = 'epoch',
    load_best_model_at_end       = True,
    metric_for_best_model        = 'f1_score',
    greater_is_better             = True,

    logging_steps                = 50,
    report_to                    = 'none',
)

print("✅ Training configuration set!")
print(f"\n   Epochs        : 10")
print(f"   Batch Size    : 32")
print(f"   Learning Rate : 3e-5")
print(f"   LR Scheduler  : cosine")
print(f"   Warmup Steps  : 500")
print(f"   Max Grad Norm : 1.0")
print(f"   Loss          : Weighted CrossEntropy")
print(f"   Best Model by : f1_score (macro)")

✅ Training configuration set!

   Epochs        : 10
   Batch Size    : 32
   Learning Rate : 3e-5
   LR Scheduler  : cosine
   Warmup Steps  : 500
   Max Grad Norm : 1.0
   Loss          : Weighted CrossEntropy
   Best Model by : f1_score (macro)


In [10]:
#═══════════════════════════════════════════
# STEP 11: Train TinyBERT
# Takes 30-40 minutes with GPU
#═══════════════════════════════════════════
trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print("=" * 55)
print("   STARTING IMPROVED TINYBERT TRAINING")
print("=" * 55)
print(f"   Model    : TinyBERT_General_4L_312D")
print(f"   Task     : Agile Requirement Prioritization")
print(f"   Classes  : High | Medium | Low")
print(f"   Samples  : {len(train_dataset):,}")
print(f"   Epochs   : 10")
print(f"   Time     : 30-40 minutes")
print("=" * 55)
print("\nTraining in progress... please wait\n")

trainer.train()

print("\n✅ Training Complete!")

   STARTING IMPROVED TINYBERT TRAINING
   Model    : TinyBERT_General_4L_312D
   Task     : Agile Requirement Prioritization
   Classes  : High | Medium | Low
   Samples  : 23,274
   Epochs   : 10
   Time     : 30-40 minutes

Training in progress... please wait



Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,F1 Weighted
1,0.655600,0.656681,0.707800,0.712900,0.704200
2,0.550000,0.585717,0.741100,0.750100,0.741100
3,0.534000,0.573211,0.726400,0.727100,0.722000
4,0.487800,0.588711,0.729100,0.732600,0.726000
5,0.420800,0.569808,0.748000,0.756000,0.747500
6,0.406100,0.574893,0.746000,0.753300,0.744500
7,0.363100,0.588457,0.746000,0.753200,0.745500
8,0.340800,0.622841,0.737700,0.744400,0.735800
9,0.305500,0.638787,0.743600,0.751000,0.742700
10,0.298900,0.638380,0.742900,0.750400,0.742000



✅ Training Complete!


In [11]:
#═══════════════════════════════════════════
# STEP 12: Save model immediately
#═══════════════════════════════════════════
import shutil
from google.colab import drive

drive.mount('/content/drive')

model.save_pretrained("./tinybert-priority-final")
tokenizer.save_pretrained("./tinybert-priority-final")
print("✅ Model saved locally!")

shutil.copytree(
    "./tinybert-priority-final",
    "/content/drive/MyDrive/tinybert-priority-final",
    dirs_exist_ok=True
)
print("✅ Model saved to Google Drive!")
print("   Location: MyDrive/tinybert-priority-final")

Mounted at /content/drive
✅ Model saved locally!
✅ Model saved to Google Drive!
   Location: MyDrive/tinybert-priority-final


In [12]:
#═══════════════════════════════════════════
# STEP 13: Evaluate on test set
#═══════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['priority_id'].tolist()

print("=" * 55)
print("         FINAL MODEL EVALUATION")
print("=" * 55)
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['High', 'Medium', 'Low']
))

cm    = confusion_matrix(true_labels, pred_labels)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual High', 'Actual Medium', 'Actual Low'],
    columns = ['Pred High',   'Pred Medium',   'Pred Low']
)
print("Confusion Matrix:")
print(cm_df)

# Compare with previous version
print("\n" + "=" * 55)
print("   COMPARISON WITH PREVIOUS VERSION")
print("=" * 55)
v1_accuracy = 0.76
v1_macro_f1 = 0.77

v2_accuracy = accuracy_score(true_labels, pred_labels)
v2_macro_f1 = f1_score(true_labels, pred_labels, average='macro')

print(f"{'Metric':<15} {'Before':<10} {'After':<10} {'Change'}")
print("-" * 50)
print(f"{'Accuracy':<15} {v1_accuracy:<10.4f} {v2_accuracy:<10.4f} "
      f"{(v2_accuracy-v1_accuracy)*100:+.1f}%")
print(f"{'Macro F1':<15} {v1_macro_f1:<10.4f} {v2_macro_f1:<10.4f} "
      f"{(v2_macro_f1-v1_macro_f1)*100:+.1f}%")

         FINAL MODEL EVALUATION
              precision    recall  f1-score   support

        High       0.69      0.85      0.76      1069
      Medium       0.82      0.68      0.74      1179
         Low       0.85      0.78      0.81       662

    accuracy                           0.76      2910
   macro avg       0.78      0.77      0.77      2910
weighted avg       0.78      0.76      0.76      2910

Confusion Matrix:
               Pred High  Pred Medium  Pred Low
Actual High          910          123        36
Actual Medium        322          798        59
Actual Low            87           57       518

   COMPARISON WITH PREVIOUS VERSION
Metric          Before     After      Change
--------------------------------------------------
Accuracy        0.7600     0.7649     +0.5%
Macro F1        0.7700     0.7715     +0.2%


In [15]:
#═══════════════════════════════════════════
# STEP 14: Rank all requirements by priority
#═══════════════════════════════════════════

def predict_priority(text):
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = 128
    )
    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=1)[0]
    pred_id    = probs.argmax().item()
    confidence = round(probs[pred_id].item() * 100, 2)
    priority_map = {0: 'High', 1: 'Medium', 2: 'Low'}
    return priority_map[pred_id], confidence, probs.tolist()

print("Ranking all requirements...")
results = []

for i, row in df_unlabeled.iterrows():
    # Access the text by its positional index (first column)
    text_content = row.iloc[0]
    priority, confidence, probs = predict_priority(text_content)
    results.append({
        'text'        : text_content,
        'source'      : 'unlabeled', # Assign a default source as it doesn't exist in df_unlabeled
        'priority'    : priority,
        'confidence'  : confidence,
        'prob_high'   : round(probs[0] * 100, 2),
        'prob_medium' : round(probs[1] * 100, 2),
        'prob_low'    : round(probs[2] * 100, 2),
    })

df_results = pd.DataFrame(results)

priority_order = {'High': 0, 'Medium': 1, 'Low': 2}
df_results['sort_priority']   = df_results['priority'].map(priority_order)
df_results['sort_confidence'] = -df_results['confidence']
df_results = df_results.sort_values(
    ['sort_priority', 'sort_confidence']
).drop(
    columns=['sort_priority', 'sort_confidence']
).reset_index(drop=True)

df_results.index      += 1
df_results.index.name  = 'implement_order'

def suggest_sprint(rank, total):
    pct = rank / total
    if pct <= 0.25:   return 'Sprint 1'
    elif pct <= 0.50: return 'Sprint 2'
    elif pct <= 0.75: return 'Sprint 3'
    else:             return 'Sprint 4'

total = len(df_results)
df_results['sprint'] = [
    suggest_sprint(i+1, total) for i in range(total)
]

print("\n" + "=" * 85)
print("      AGILE REQUIREMENTS — IMPLEMENTATION ORDER")
print("=" * 85)
print(f"{'Rank':<6} {'Priority':<10} {'Sprint':<12} {'Confidence':<13} Requirement")
print("-" * 85)

for idx, row in df_results.iterrows():
    emoji = {'High':'🔴','Medium':'🟡','Low':'🟢'}[row['priority']]
    print(f"{idx:<6} {emoji} {row['priority']:<8} "
          f"{row['sprint']:<12} {row['confidence']}%"
          f"{'':6} {row['text'][:50]}...")

print("\n" + "=" * 85)
print("📊 SUMMARY:")
print(f"   🔴 High   : {len(df_results[df_results['priority']=='High']):>3} → Sprint 1")
print(f"   🟡 Medium : {len(df_results[df_results['priority']=='Medium']):>3} → Sprint 2-3")
print(f"   🟢 Low    : {len(df_results[df_results['priority']=='Low']):>3} → Sprint 4")
print("=" * 85)

Ranking all requirements...

      AGILE REQUIREMENTS — IMPLEMENTATION ORDER
Rank   Priority   Sprint       Confidence    Requirement
-------------------------------------------------------------------------------------
1      🔴 High     Sprint 1     96.46%       As an Owner, I want to reset the environment to on...
2      🔴 High     Sprint 1     93.51%       As a Developer, I want D Files generation requests...
3      🔴 High     Sprint 1     92.33%       As an owner, I want to be sure that USAspending on...
4      🔴 High     Sprint 1     89.86%       As an agency user, I want the FABS validation rule...
5      🔴 High     Sprint 1     88.65%       As a FABS user, I want to have read-only access to...
6      🔴 High     Sprint 1     87.63%       As an Agency user, I want to receive a more helpfu...
7      🔴 High     Sprint 1     84.29%       As an agency user, I want the FABS validation rule...
8      🔴 High     Sprint 1     80.35%       As an agency user, I want to be confident that the

In [16]:
#═══════════════════════════════════════════
# STEP 15: Download final results
#═══════════════════════════════════════════
from google.colab import files

df_results.to_csv("agile_requirements_ranked.csv")
df_results.to_csv("/content/drive/MyDrive/agile_requirements_ranked.csv")

files.download("agile_requirements_ranked.csv")

print("✅ Downloaded: agile_requirements_ranked.csv")
print(f"\n🎯 Research Complete!")
print(f"   Model    : TinyBERT (fine-tuned)")
print(f"   Accuracy : {v2_accuracy:.2%}")
print(f"   Macro F1 : {v2_macro_f1:.4f}")
print(f"   {len(df_results)} requirements ranked #1 to #{len(df_results)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: agile_requirements_ranked.csv

🎯 Research Complete!
   Model    : TinyBERT (fine-tuned)
   Accuracy : 76.49%
   Macro F1 : 0.7715
   97 requirements ranked #1 to #97
